<a href="https://colab.research.google.com/github/speediedan/finetuning-scheduler/blob/main/src/fts_examples/notebooks/publish/superglue/fts_superglue_pytorch.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

In [ ]:
# To run this notebook in a fresh environment (Colab, for example), uncomment the line below. The PyTorch API
# needs Fine-Tuning Scheduler 2.15, installed from main until 2.15 is released.
# %pip install 'finetuning-scheduler[examples] @ git+https://github.com/speediedan/finetuning-scheduler.git@main'


# Scheduled Fine-Tuning with the Fine-Tuning Scheduler PyTorch API

![Fine-Tuning Scheduler logo](logo_fts.png){height="55px" width="401px"}

[Fine-Tuning Scheduler](https://finetuning-scheduler.readthedocs.io/en/stable/index.html) (FTS) accelerates and enhances
foundation model experimentation with flexible, multi-phase fine-tuning schedules. A schedule says which parameters to
thaw in each phase; FTS thaws them, adds them to the optimizer's parameter groups (optionally reinitializing the
optimizer or lr scheduler) and decides when each phase ends. Training with FTS is simple and brings several benefits:

- it conveniently abstracts and composes a broad space of fine-tuning methods
- expedites and facilitates exploration of model tuning dynamics
- it composes with mechanistic interpretability research packages, through [tunable views](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/tunable_views.html), enabling latent
  space tuning and expanding the space of experimental interventions

FTS runs a schedule through either of two interfaces, which share one scheduling core and one schedule format: the
[FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) callback for Lightning, and the PyTorch API's [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) for a training loop you write yourself. This notebook uses
the PyTorch API, which is experimental in Fine-Tuning Scheduler 2.15, and does not import Lightning. The
[Lightning version of this tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/superglue/fts_superglue_lightning.html) runs the same experiment with the callback.

Setup is straightforward, just install from PyPI! Since this notebook-based example requires a few additional packages
(e.g. ``transformers``, ``sentencepiece``), install the ``finetuning-scheduler`` package with the ``[examples]`` extra.
For additional installation options, please see the Fine-Tuning Scheduler
[README](https://github.com/speediedan/finetuning-scheduler/blob/main/README.md).



<div style="display:inline">

Fundamentally, [Fine-Tuning Scheduler](https://finetuning-scheduler.readthedocs.io/en/stable/index.html) enables
scheduled, multi-phase, fine-tuning of foundation models. Gradual unfreezing (i.e. thawing) can help maximize
foundation model knowledge retention while allowing (typically upper layers of) the model to
optimally adapt to new tasks during transfer learning [1, 2, 3]

</div>

A [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) orchestrates the gradual unfreezing of a model via a fine-tuning schedule that is either implicitly generated
(the default) or explicitly provided by the user (more computationally efficient). It owns which parameters are
trainable, the optimizer's parameter groups and when the next phase starts; the loop stays yours. Phase transitions are
driven by transition policies such as [PlateauTransition](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.PlateauTransition) (the PyTorch API's counterpart of the callback's [FTSEarlyStopping](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSEarlyStopping)), by the epoch
limit a phase may set (``max_transition_epoch``), or by a composition of the two. A training session completes when the
final phase of the schedule has its stopping criteria met.

![Fine-Tuning Scheduler explicit loss animation](fts_explicit_loss_anim.gif){height="272px" width="376px"}

## Basic Usage

If no fine-tuning schedule is provided (``None``), the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) generates a
[default schedule](#The-Default-Fine-Tuning-Schedule) and fine-tunes according to it. Build it for your model, call
``setup()`` once, iterate over ``epochs()``, and report each epoch's validation metrics with ``epoch_end()``. It builds
the optimizer from the first phase's parameters and decides when each phase starts; the loop stays yours:

```python
from functools import partial

import torch
from finetuning_scheduler import PlateauTransition, TuningScheduler

fts = TuningScheduler(
    model,
    None,  # the default schedule, or the path to a schedule YAML file, or a mapping
    optimizer=partial(torch.optim.AdamW, lr=1e-3),
    transition=PlateauTransition(monitor="val_loss", patience=2, min_delta=0.01),
)
fts.setup()
for epoch in fts.epochs(max_epochs=30):
    for x, y in train_loader:
        loss = torch.nn.functional.cross_entropy(model(x), y)
        loss.backward()
        fts.optimizer.step()  # always `fts.optimizer`: a phase can replace the optimizer
        fts.optimizer.zero_grad()
    fts.epoch_end({"val_loss": evaluate(model)})
```

## The Default Fine-Tuning Schedule

Schedule definition is facilitated via [gen_ft_schedule](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.schedule.html#finetuning_scheduler.core.schedule.gen_ft_schedule), which dumps a default fine-tuning schedule (by default using a naive,
2-parameters per level heuristic) which can be adjusted as desired by the user and/or subsequently passed to the
[TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler). Using the default/implicitly generated schedule will likely be less computationally efficient than a
user-defined fine-tuning schedule but is useful for exploring a model's fine-tuning behavior and can serve as a good
baseline for subsequent explicit schedule refinement. While the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) supports a single optimizer and (optional) lr
scheduler, per-phase maximum learning rates can be set as demonstrated in the next section.

## Specifying a Fine-Tuning Schedule

To specify a fine-tuning schedule, it's convenient to first generate the default schedule and then alter the
thawed/unfrozen parameter groups associated with each fine-tuning phase as desired. Fine-tuning phases are zero-indexed
and executed in ascending order.

1. First, generate the default schedule to a directory of your choice. It will be named after your model's class with
   the suffix ``_ft_schedule.yaml``.

```python
from finetuning_scheduler.core.schedule import gen_ft_schedule

schedule_path = gen_ft_schedule(model, "schedules/")
```

2. Alter the schedule as desired.

![side_by_side_yaml](side_by_side_yaml.png){height="327px" width="800px"}

3. Once the fine-tuning schedule has been altered as desired, pass it to the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) to commence scheduled training (a
   mapping with the same content works too):

```python
fts = TuningScheduler(model, "/path/to/my/schedule/my_schedule.yaml", optimizer=..., transition=...)
```

A schedule names parameters as the model does: a [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) is given the model itself, so its schedules have no ``model.``
prefix, which the Lightning tutorial's schedules carry because there the model is an attribute of a
``LightningModule``.

## Early-Stopping and Epoch-Driven Phase Transition Criteria

Early-stopping (plateau) and epoch-driven transition criteria are composed, as they are under Lightning: a
[PlateauTransition](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.PlateauTransition) is the PyTorch API's counterpart of the callback's ``FTSEarlyStopping``. If a ``max_transition_epoch`` is specified for a
given phase, the next fine-tuning phase will begin at that epoch unless the early-stopping criteria (a plateau in
``val_loss``, say) are met first. For transitions that are exclusively epoch-driven (the callback's
``epoch_transitions_only``), pass no plateau policy: every phase but the final one then sets its
``max_transition_epoch``, and the final phase runs to ``max_epochs`` (or to an [EpochTransition](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.EpochTransition)). Several policies can be
passed as a list or composed with [AnyOf](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.AnyOf), and a policy can also end a phase at a step count or on any condition you
write (see [Transitions](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/plain_pytorch_loop.html#transitions)).

<div class="alert alert-info">

**Tip:** Use of regex expressions can be convenient for specifying more complex schedules. Also, a per-phase base
maximum lr can be specified:

<pre>
0:
<mark>  params: # the parameters for each phase definition can be fully specified</mark>
  - classifier.bias
  - classifier.weight
  max_transition_epoch: 3
1:
<mark>  params: # or specified via a regex</mark>
  - albert.pooler.*
2:
  params:
  - albert.encoder.*.ffn_output.*
  max_transition_epoch: 9
<mark>  lr: 1e-06 # per-phase maximum learning rates can be specified</mark>
3:
<mark>  params: # both approaches to parameter specification can be used in the same phase</mark>
  - albert.encoder.*.(ffn\.|attention|full*).*
  - albert.encoder.embedding_hidden_mapping_in.bias
  - albert.encoder.embedding_hidden_mapping_in.weight
  - albert.embeddings.*
</pre>

</div>

The end-to-end example in this notebook ([Scheduled Fine-Tuning For SuperGLUE](#Scheduled-Fine-Tuning-For-SuperGLUE))
uses the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) in explicit mode to fine-tune a small foundation model on the
[RTE](https://huggingface.co/datasets/aps/super_glue) task of [SuperGLUE](https://super.gluebenchmark.com/).

## Resuming Scheduled Fine-Tuning Training Sessions

``save()`` writes the full training state for an exact resume: the model, optimizer and lr scheduler, the
[TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler)'s own state (its phase, its transition policies' state and its checkpoint bookkeeping), every random stream, and
anything registered with ``register_stateful`` (a sampler or a data position, for example). To resume, build the same
model, schedule and [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) in a fresh process, then call ``setup()`` and ``load()``:

```python
fts.register_stateful("sampler", sampler)  # anything with state_dict() and load_state_dict()
fts.save("some/path/to/my_state.pt")

# later, in a fresh process, after building the same model, schedule and tuning scheduler
fts.setup()
fts.load("some/path/to/my_state.pt")
for epoch in fts.epochs(max_epochs=30):  # continues from the saved phase and epoch
    ...
```

Training will resume at the depth/level of the saved state according to the specified schedule. Schedules can be
altered between training sessions but schedule compatibility is left to the user for maximal flexibility. If executing
a user-defined schedule, typically the same schedule should be provided for the original and resumed training sessions.

By default (``restore_best`` is ``True`` whenever a checkpoint policy such as [BestCheckpoint](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.checkpoint.html#finetuning_scheduler.core.checkpoint.BestCheckpoint) is given), the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) restores the
best available checkpoint before each fine-tuning depth transition, so every phase starts from the best state of the
one before.

A save keeps the checkpoints it refers to until the next save, so resuming from the latest save always finds them. An
older save can be resumed while its checkpoints remain; if its best checkpoint has been removed, ``load()`` raises
``FileNotFoundError`` rather than restoring the wrong state later.

<div class="alert alert-warning">

**Note:** The [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) schedules models you shard yourself, before handing them to it: FSDP2 (``fully_shard``), tensor
parallelism (``parallelize_module``) or both, calling it on every rank (see
[Distributed Training](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/plain_pytorch_loop.html#distributed-training) and the [model-parallel example](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/model_parallel_plain_loop.html)).
``DistributedDataParallel`` (and the composable ``replicate()``) and ``ZeroRedundancyOptimizer`` are refused: DDP
registers only the parameters that require grad when it is built, so parameters thawed in later phases would never be
gradient-synchronized. With FSDP2, they stay synchronized.
</div>

## Scheduled Fine-Tuning For SuperGLUE

The following example demonstrates the use of the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) to fine-tune a small foundation model on the
[RTE](https://huggingface.co/datasets/aps/super_glue) task of [SuperGLUE](https://super.gluebenchmark.com/). It runs the
same experiment as the [Lightning tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/superglue/fts_superglue_lightning.html), in a training loop of our own. Iterative early-stopping
will be applied according to a user-specified schedule. Every run logs to TensorBoard under the same metric names the
Lightning tutorial's ``TensorBoardLogger`` uses, so the two tutorials' training curves can be compared directly.

In [ ]:
# papermill parameters: shrink these for a quick smoke run
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_EPOCHS = 100
TRAIN_SUBSET = None  # an int keeps only that many training examples
VAL_SUBSET = None
BATCH_SIZE = 16
SEED = 42
MAX_DEPTH = 1  # how deep the fts_explicit and fts_implicit schedules go; -1 runs every phase (the docs render's)

In [ ]:
import os
import warnings
from functools import partial

import datasets
import torch
import yaml
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tensorboardX import SummaryWriter
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import logging as transformers_logging

from finetuning_scheduler import PlateauTransition, TuningScheduler
from finetuning_scheduler.core.checkpoint import BestCheckpoint
from finetuning_scheduler.core.schedule import gen_ft_schedule

transformers_logging.set_verbosity_error()
datasets.logging.set_verbosity_error()
datasets.logging.disable_progress_bar()
warnings.filterwarnings("ignore", r".*converting to a fast.*")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Running on {device}" + ("" if use_amp else " (this notebook is best run with a GPU)"))

### Data

RTE pairs a premise with a hypothesis and asks whether the premise entails it. We tokenize the pairs once and build
ordinary ``DataLoader``s.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, clean_up_tokenization_spaces=True)
raw = datasets.load_dataset("aps/super_glue", "rte")


def encode(batch):
    features = tokenizer(batch["premise"], batch["hypothesis"], max_length=128, padding="max_length", truncation=True)
    features["labels"] = batch["label"]
    return features


def dataset(split: str, subset: int | None):
    ds = raw[split] if subset is None else raw[split].select(range(subset))
    ds = ds.map(encode, batched=True, remove_columns=ds.column_names)
    ds.set_format(type="torch")
    return ds


def train_loader() -> DataLoader:
    # a seeded shuffle, made afresh for each run: every scenario sees the same order, whatever ran before it
    return DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))


train_ds = dataset("train", TRAIN_SUBSET)
val_loader = DataLoader(dataset("validation", VAL_SUBSET), batch_size=BATCH_SIZE)
print(f"{len(train_ds)} training and {len(val_loader.dataset)} validation examples")

### Our Training Sessions

We'll be comparing three different fine-tuning training configurations. Every configuration in this example depends
upon a shared set of defaults, only differing in their respective fine-tuning schedules.

| Experiment Tag    | Training Scenario Description                                          |
|:-----------------:| ---------------------------------------------------------------------- |
| ``fts_explicit``  | Training with a fine-tuning schedule explicitly provided by the user     |
| ``nofts_baseline``| A baseline fine-tuning training session (without scheduled fine-tuning)  |
| ``fts_implicit``  | Training with an implicitly generated fine-tuning schedule (the default) |

Let's begin by generating the default schedule for our model, as step 1 of
[Specifying a Fine-Tuning Schedule](#Specifying-a-Fine-Tuning-Schedule) does, and looking at its first phases.

In [ ]:
schedule_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, dtype=torch.float32)
default_schedule_path = gen_ft_schedule(schedule_model, os.path.join("fts_logs", "generated"))
del schedule_model
with open(default_schedule_path) as f:
    default_schedule = yaml.safe_load(f)
print(f"{len(default_schedule)} phases in {default_schedule_path}, the first three:\n")
print(yaml.safe_dump({k: default_schedule[k] for k in list(default_schedule)[:3]}, sort_keys=False))

The default schedule thaws two parameters per phase from the last registered one: the classifier head first, then the
pooler, then each encoder layer's parameters, pair by pair, ending with the embeddings. That many phases is useful for
exploring the model's tuning dynamics (the ``fts_implicit`` scenario uses it), but for ``fts_explicit`` we alter it into
three phases, as the Lightning tutorial does. Phase 0 thaws the classifier head, the pooler, the encoder's
``LayerNorm`` and relative position embeddings, and the attention, intermediate and output blocks of every encoder
layer (``.{0,11}`` matches any layer index); phase 1 thaws the embedding ``LayerNorm``; phase 2 the word embeddings.
Entries are regular expressions, and a phase may also set its own base learning rate with ``lr``. As in the Lightning
tutorial, we write the schedule to a file to simulate loading an explicitly defined fine-tuning schedule.

In [ ]:
# Please see the [Fine-Tuning Scheduler documentation](https://finetuning-scheduler.readthedocs.io/en/stable/index.html)
# for a full description of the schedule format
ft_schedule_yaml = """
0:
  params:
  - classifier.bias
  - classifier.weight
  - pooler.dense.bias
  - pooler.dense.weight
  - deberta.encoder.LayerNorm.bias
  - deberta.encoder.LayerNorm.weight
  - deberta.encoder.rel_embeddings.weight
  - deberta.encoder.layer.{0,11}.(output|attention|intermediate).*
1:
  params:
  - deberta.embeddings.LayerNorm.bias
  - deberta.embeddings.LayerNorm.weight
2:
  params:
  - deberta.embeddings.word_embeddings.weight
"""
os.makedirs("fts_logs", exist_ok=True)
explicit_schedule = os.path.join("fts_logs", "DebertaV2ForSequenceClassification_ft_schedule_explicit.yaml")
with open(explicit_schedule, "w") as f:
    f.write(ft_schedule_yaml)
baseline_schedule = {0: {"params": [".*"]}}  # a mapping works too: one phase, every parameter trainable from the start

### Optimizer Configuration

Though other optimizers can arguably yield some marginal advantage contingent on the context,
the Adam optimizer (and the [AdamW version](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html) which
implements decoupled weight decay) remains robust to hyperparameter choices and is commonly used for fine-tuning
foundation language models. See (Sivaprasad et al., 2020) and (Mosbach, Andriushchenko & Klakow, 2020) for theoretical
and systematic empirical justifications of Adam and its use in fine-tuning large transformer-based language models. The
values used here have some justification in the referenced literature but have been largely empirically determined
and while a good starting point could be further tuned.

The optimizer is passed to the tuning scheduler as a factory: it builds the optimizer from phase 0's parameter groups
and adds a group as each later phase thaws parameters.

In [ ]:
optimizer_factory = partial(AdamW, weight_decay=1e-05, eps=1e-07, lr=1e-05)

### LR Scheduler Configuration

The [CosineAnnealingWarmRestarts scheduler](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.CosineAnnealingWarmRestarts.html)
nicely fits with our iterative fine-tuning since it does not depend upon a global max_epoch value. The importance of
initial warmup is reduced due to the innate warmup effect of Adam bias correction [5] and the gradual thawing we are
performing. Note that commonly used LR schedulers that depend on providing max_iterations/epochs (e.g. the
[CosineWarmupScheduler](https://github.com/Lightning-AI/tutorials/blob/0c325829101d5a6ebf32ed99bbf5b09badf04a59/course_UvA-DL/05-transformers-and-MH-attention/Transformers_MHAttention.py#L688)
used in other tutorials) also work with Fine-Tuning Scheduler. Though the LR scheduler is theoretically justified
(Loshchilov & Hutter, 2016), the particular values provided here are primarily empirically driven.

Like the optimizer, the lr scheduler is passed as a factory, here stepped once per epoch. The tuning scheduler also
supports both optimizer and LR scheduler reinitialization in explicit and implicit fine-tuning schedule modes. See the
advanced usage documentation
([LR scheduler reinitialization](https://finetuning-scheduler.readthedocs.io/en/latest/advanced/lr_scheduler_reinitialization.html),
[optimizer reinitialization](https://finetuning-scheduler.readthedocs.io/en/latest/advanced/optimizer_reinitialization.html))
for explanations and demonstration of the extension's support for more complex requirements.

In [ ]:
lr_scheduler_factory = partial(CosineAnnealingWarmRestarts, T_0=1, T_mult=2, eta_min=1e-07)

### Transitions and Checkpoints

[PlateauTransition](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.PlateauTransition) plays the part ``FTSEarlyStopping`` plays under Lightning, with the Lightning tutorial's configuration: a
phase ends when ``val_loss`` has not improved by ``min_delta`` for ``patience`` epochs. [BestCheckpoint](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.checkpoint.html#finetuning_scheduler.core.checkpoint.BestCheckpoint) keeps the best
checkpoint, and the tuning scheduler restores it at each phase boundary, so every phase starts from the best state of
the one before.

In [ ]:
def make_tuning_scheduler(model, schedule, scenario: str, max_depth: int) -> TuningScheduler:
    return TuningScheduler(
        model,
        schedule,
        optimizer=optimizer_factory,
        lr_scheduler=lr_scheduler_factory,
        lr_scheduler_cfg={"interval": "epoch"},
        transition=PlateauTransition(monitor="val_loss", patience=2, min_delta=0.001),
        checkpoint=BestCheckpoint(os.path.join("checkpoints", scenario), monitor="val_loss"),
        max_depth=max_depth,
        log_dir=os.path.join("fts_logs", scenario),
    )

### The Training Loop

This is an ordinary mixed-precision loop. The tuning scheduler appears in four places: ``setup()`` before training,
``epochs()`` to iterate (it starts the next phase when one is due), ``fts.optimizer`` in place of an optimizer held by
the loop (a schedule may replace the optimizer at a phase transition), and ``epoch_end()`` with the epoch's
validation metrics.

Each run seeds itself where its model is built and draws a freshly seeded shuffle of the training data, so no scenario
depends on the ones run before it, and the model trains in train mode (``from_pretrained`` returns it in eval mode,
which would leave dropout off). The Lightning tutorial is configured the same way, so the two tutorials' runs can be
compared directly.

Each run also writes TensorBoard logs to ``tensorboard_logs/<scenario>``, with the Lightning tutorial's metric
names: ``train_loss`` at every optimizer step, and ``val_loss``, ``accuracy`` and ``finetuning_schedule_depth`` at
the end of each epoch, plus each parameter group's learning rate and the number of trainable parameters, all
against the optimizer step. A ``SummaryWriter`` is all it takes: ``tensorboardX`` is the writer Lightning's
``TensorBoardLogger`` uses when ``tensorboard`` itself is not installed.

In [ ]:
@torch.no_grad()
def evaluate(model) -> dict[str, float]:
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            out = model(**batch)
        n = batch["labels"].numel()
        total_loss += out.loss.float().item() * n
        correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
        count += n
    return {"val_loss": total_loss / count, "accuracy": correct / count}


def run(scenario: str, schedule, max_depth: int) -> list[dict]:
    torch.manual_seed(SEED)  # seed each scenario where its model is built, so none depends on the ones before it
    # fp32 master weights: transformers would otherwise keep the checkpoint's stored dtype, and autocast handles fp16
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, dtype=torch.float32)
    model = model.to(device)
    fts = make_tuning_scheduler(model, schedule, scenario, max_depth)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    history = []
    writer = SummaryWriter(os.path.join("tensorboard_logs", scenario))
    global_step, loader = 0, train_loader()
    fts.setup()
    for epoch in fts.epochs(max_epochs=MAX_EPOCHS):
        train_loss, steps = 0.0, 0
        model.train()  # `from_pretrained` returns the model in eval mode, and `evaluate` leaves it there
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            scaler.step(fts.optimizer)
            scaler.update()
            fts.optimizer.zero_grad()
            train_loss, steps, global_step = train_loss + loss.item(), steps + 1, global_step + 1
            writer.add_scalar("train_loss", loss.item(), global_step)
        metrics = evaluate(model)
        fts.step_lr_schedulers("epoch")
        fts.epoch_end({"val_loss": metrics["val_loss"]})
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        history.append({"epoch": epoch, "depth": fts.curr_depth, "trainable": trainable, **metrics})
        for name, value in (*metrics.items(), ("finetuning_schedule_depth", fts.curr_depth),
                            ("trainable_params", trainable)):
            writer.add_scalar(name, value, global_step)
        for i, group in enumerate(fts.optimizer.param_groups):
            writer.add_scalar(f"lr-AdamW/pg{i + 1}", group["lr"], global_step)
        print(
            f"{scenario:>14} | epoch {epoch:>2} | phase {fts.curr_depth} | trainable {trainable:>11,} | "
            f"lr {fts.optimizer.param_groups[0]['lr']:.2e} | train_loss {train_loss / steps:.4f} | "
            f"val_loss {metrics['val_loss']:.4f} | accuracy {metrics['accuracy']:.4f}"
        )
    writer.close()
    print(f"{scenario}: finished at phase {fts.curr_depth} after {len(history)} epochs ({fts.stop_reason or 'done'})")
    return history

### Running the Scenarios

``fts_explicit`` uses the schedule above. ``fts_implicit`` passes no schedule, so the tuning scheduler generates the
default one we generated earlier (parameters thawed in pairs, from the last registered parameter). ``MAX_DEPTH = 1``
keeps both to their first two phases, which shows the phase mechanics at a reasonable cost, as the Lightning tutorial's
``max_depth=1`` does; set it to ``-1`` to run every phase. The baseline's single phase makes it ordinary fine-tuning
with early stopping.

The copy of this notebook rendered in the documentation was run with ``MAX_DEPTH = -1``, so its ``fts_implicit`` run
goes as deep as ``MAX_EPOCHS`` allows (about half an hour on one recent GPU in all).

In [ ]:
# optionally start TensorBoard and watch the runs' curves while they train by uncommenting the next 2 lines
# %load_ext tensorboard
# %tensorboard --logdir tensorboard_logs

In [ ]:
results = {
    "fts_explicit": run("fts_explicit", explicit_schedule, MAX_DEPTH),
    "fts_implicit": run("fts_implicit", None, MAX_DEPTH),
    "nofts_baseline": run("nofts_baseline", baseline_schedule, -1),
}

### Reviewing the Training Trajectory

The TensorBoard logs record each run's curves against the optimizer step. The run of this notebook rendered in the
documentation (one GPU, ``MAX_DEPTH = -1``) trained as follows, from its per-epoch lines above (the best ``val_loss`` of
each scenario in bold); expect some variation with the hardware and package versions:

| Scenario | Phase | Thaws | Epochs | Trainable parameters | ``val_loss`` by epoch |
| --- | --- | --- | --- | --- | --- |
| ``fts_explicit`` | 0 | the classifier head, the pooler, the encoder's ``LayerNorm`` and relative position embeddings, and every encoder layer's attention, intermediate and output blocks | 0 to 3 | 86,041,346 | 0.648, **0.392**, 0.460, 0.726 |
| | 1 | adds the embedding ``LayerNorm`` | 4 and 5 | 86,042,882 | 0.491, 0.554 |
| | 2 | adds the word embeddings | 6 and 7 | 184,423,682 | 0.420, 0.521 |
| ``nofts_baseline`` | (one) | every parameter | 0 to 3 | 184,423,682 | 0.648, **0.392**, 0.463, 0.747 |
| ``fts_implicit`` | 0 to 9 | the classifier head, the pooler, the encoder's ``LayerNorm`` and relative position embeddings, then the last encoder layer two parameters at a time | 0 to 30 | 1,538 to 6,894,338 | 0.697 falling to 0.684 |
| | 10 | adds the last encoder layer's attention key projection (and its query projection's bias) | 31 to 40 | 7,484,930 | 0.681 falling to 0.471 |
| | 11 to 31 | two parameters at a time, on down to encoder layer 8 (the generated schedule has 101 phases, and the run reaches ``MAX_EPOCHS`` first) | 41 to 99 | 8,075,522 to 26,976,770 | between **0.368** (epoch 81) and 0.571 |

Validation loss and the schedule depth for ``fts_explicit`` show its transitions. When the plateau ends phase 0, the
tuning scheduler restores the best checkpoint before phase 1 thaws the embedding ``LayerNorm``, so training resumes
from the best state rather than the overfit one, and validation loss falls back from 0.726 to 0.491; phase 2 then
thaws the word embeddings and brings it to 0.420, and at the final phase the plateau ends the run after epoch 7:

![pytorch_transition](pytorch_fts_explicit_transition.png){width="976px"}

``nofts_baseline`` trains every parameter from the start, so the plateau ends it after four epochs.

[Fine-Tuning Scheduler](https://finetuning-scheduler.readthedocs.io/en/stable/index.html) expands the space of possible
fine-tuning schedules and the composition of more sophisticated schedules can yield marginal fine-tuning performance
gains. That stated, it should be emphasized the primary utility of Fine-Tuning Scheduler is to grant greater fine-tuning
flexibility for model exploration in research. For example, glancing at DeBERTa-v3's implicit training run, a critical
tuning transition point is immediately apparent:

![pytorch_implicit_training_transition](pytorch_implicit_training_transition.png){width="976px"}

Our ``val_loss`` stays near 0.69 (chance level for RTE) through the first ten phases, which thaw the classifier head,
the pooler and the last encoder layer's parameters a pair at a time, then begins a precipitous decline at about step
5,000, which corresponds to phase 10 in the schedule. Referring to the
[generated schedule](#Our-Training-Sessions), in phase 10 we begin tuning the key projection of the last encoder
layer's attention (``deberta.encoder.layer.11``, counting from 0), whose value projection has been trainable since
phases 8 and 9: from there the layer can change what it attends to, not only what it passes on. Interesting! Though
beyond the scope of this tutorial, it might be worth investigating these dynamics further, and the tuning scheduler
allows one to do just that quite easily. The implicit run also reaches the lowest ``val_loss`` of the three scenarios
(0.368, at epoch 81), at the cost of a hundred epochs.

The Lightning tutorial's implicit run, from an older multi-GPU run, shows the same kind of transition at a later
phase: where it falls depends on the schedule's granularity, the effective batch size and the run's random state.

#### The two tutorials side by side

The [Lightning version of this tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/superglue/fts_superglue_lightning.html) runs the same scenarios with the same model, data, optimizer,
lr scheduler, transition criteria and seed, and its ``nofts_baseline`` reproduces this notebook's epoch for epoch
(validation loss 0.648, 0.392, 0.463 and 0.747). Both tutorials seed each scenario where its model is built, shuffle
the training data with a seeded generator, and train the model in train mode, so neither scenario depends on the ones
run before it. Its ``fts_implicit`` run matches this notebook's for all eight epochs it runs (the Lightning tutorial
keeps both scheduled scenarios to ``max_depth=1``), and its ``fts_explicit`` run matches through phase 0, after which it
differs by configuration: the Lightning tutorial does not restore the best checkpoint at phase boundaries
(``restore_best=False``), so that its tests need not write checkpoints.

Expect run-to-run variation: compare scenarios over several seeds (``SEED`` in the parameters cell) before reading much
into a small difference.

To view your own run's curves, run ``tensorboard --logdir tensorboard_logs`` (``tensorboard`` installs separately) or
uncomment the cell above.


## Where to Go Next

- [Scheduled fine-tuning in a plain PyTorch loop](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/plain_pytorch_loop.html):
  transition policies, phase hooks and gates, exact resumption and distributed training.
- [Model-parallel scheduled fine-tuning in a plain loop](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/model_parallel_plain_loop.html):
  the same tuning scheduler with FSDP2 and tensor parallelism.
- The [Lightning version of this tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/superglue/fts_superglue_lightning.html), for the
  ``FinetuningScheduler`` callback.

## Footnotes

- [Howard, J., & Ruder, S. (2018)](https://arxiv.org/pdf/1801.06146.pdf). Fine-tuned Language
 Models for Text Classification. ArXiv, abs/1801.06146. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-PyTorch-API)
- [Chronopoulou, A., Baziotis, C., & Potamianos, A. (2019)](https://arxiv.org/pdf/1902.10547.pdf).
 An embarrassingly simple approach for transfer learning from pretrained language models. arXiv
 preprint arXiv:1902.10547. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-PyTorch-API)
- [Peters, M. E., Ruder, S., & Smith, N. A. (2019)](https://arxiv.org/pdf/1903.05987.pdf). To tune or not to
 tune? adapting pretrained representations to diverse tasks. arXiv preprint arXiv:1903.05987. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-PyTorch-API)
- [Sivaprasad, P. T., Mai, F., Vogels, T., Jaggi, M., & Fleuret, F. (2020)](https://arxiv.org/pdf/1910.11758.pdf).
 Optimizer benchmarking needs to account for hyperparameter tuning. In International Conference on Machine Learning
(pp. 9036-9045). PMLR. [↩](#Optimizer-Configuration)
- [Mosbach, M., Andriushchenko, M., & Klakow, D. (2020)](https://arxiv.org/pdf/2006.04884.pdf). On the stability of
fine-tuning bert: Misconceptions, explanations, and strong baselines. arXiv preprint arXiv:2006.04884. [↩](#Optimizer-Configuration)
- [Loshchilov, I., & Hutter, F. (2016)](https://arxiv.org/pdf/1608.03983.pdf). Sgdr: Stochastic gradient descent with
warm restarts. arXiv preprint arXiv:1608.03983. [↩](#LR-Scheduler-Configuration)